# Seq2Seq 기계 번역 (v1)

이번 프로젝트에선 임의로 Seq2Seq 모델을 아주 간단화 시켰습니다.  
한 언어로 된 문장을 다른 언어로 된 문장으로 번역하는 덩치가 큰 모델이 아닌  
영어 문장을 스페인어 문장으로 번역하는 Mini Seq2Seq 모델을 같이 구현해 보겠습니다.

**v1 개선사항:**
- 다층 GRU (2 layers) + Dropout
- Attention 메커니즘 추가
- 복수의 학습 데이터 쌍 사용
- 가중치 초기화 (Xavier)
- 학습률 스케줄러 (ReduceLROnPlateau)
- Gradient Clipping

## 1. 라이브러리 임포트 및 학습 데이터 정의

In [ ]:
import torch  # 파이토치 임포트
import torch.nn as nn  # 신경망 모듈 임포트
import torch.nn.functional as F  # 신경망 함수 모듈 임포트
import random  # 무작위성을 위한 random 임포트
import matplotlib.pyplot as plt  # 시각화용 matplotlib 임포트

# 학습 데이터 정의
training_pairs = [  # 영어-스페인어 학습 쌍
    ("I go to bed.", "Me voy a la cama."),
    ("I want to sleep.", "Quiero dormir."),
    ("I play a game.", "Juego un juego.")
]

vocab_size = 256  # 총 아스키 코드 개수 (단어장 크기)

print("학습 데이터:")  # 출력
for en, es in training_pairs:  # 데이터 확인 반복
    print(f"  {en} -> {es}")  # 쌍 출력

## 2. Seq2Seq 모델 정의

- **Encoder**: 2층 GRU로 입력 문장을 고정 크기 벡터로 압축
- **Attention**: 디코더가 인코더 출력의 어느 부분에 집중할지 계산
- **Decoder**: 2층 GRU + Attention으로 출력 문장을 생성

In [ ]:
class Seq2Seq(nn.Module):  # Seq2Seq 모델 정의
    def __init__(self, vocab_size, hidden_size):  # 초기화
        super(Seq2Seq, self).__init__()  # 부모 클래스 초기화
        self.n_layers = 2  # GRU 계층 수
        self.hidden_size = hidden_size  # 은닉층 크기
        self.embedding = nn.Embedding(vocab_size, hidden_size)  # 임베딩 계층
        self.encoder = nn.GRU(hidden_size, hidden_size, num_layers=self.n_layers, dropout=0.2)  # 인코더 GRU
        self.decoder = nn.GRU(hidden_size, hidden_size, num_layers=self.n_layers, dropout=0.2)  # 디코더 GRU
        self.attention = nn.Linear(hidden_size * 2, hidden_size)  # 어텐션용 선형 계층
        self.project = nn.Linear(hidden_size * 2, vocab_size)  # 출력 프로젝션 계층
        
        # 가중치 초기화
        self._init_weights()  # 초기화 메서드 호출
    
    def _init_weights(self):  # 가중치 초기화 로직
        for name, param in self.named_parameters():  # 모든 파라미터 반복
            if 'weight' in name:  # 가중치인 경우
                nn.init.xavier_uniform_(param)  # Xavier 초기화
            elif 'bias' in name:  # 편향인 경우
                nn.init.zeros_(param)  # 0으로 초기화

    def forward(self, inputs, targets):  # 순전파
        batch_size = 1  # 배치 크기 고정
        seq_len = inputs.size(0)  # 입력 시퀀스 길이
        
        # 인코더
        initial_state = self._init_state(batch_size)  # 초기 상태 생성
        embedding = self.embedding(inputs).unsqueeze(1)  # 입력 임베딩
        encoder_output, encoder_state = self.encoder(embedding, initial_state)  # 인코더 실행
        
        # 디코더
        decoder_state = encoder_state  # 디코더 초기 상태 설정
        decoder_input = torch.LongTensor([ord('V')]).to(inputs.device)  # 시작 토큰(V)
        
        outputs = []  # 출력 저장용 리스트
        for i in range(targets.size(0)):  # 타겟 길이만큼 반복
            decoder_input = self.embedding(decoder_input).unsqueeze(1)  # 디코더 입력 임베딩
            decoder_output, decoder_state = self.decoder(decoder_input, decoder_state)  # 디코더 실행
            
            # 어텐션 계산
            decoder_output_reshaped = decoder_output.squeeze(1)  # 디코더 출력 정렬
            encoder_output_reshaped = encoder_output.squeeze(1)  # 인코더 출력 정렬
            
            attention_scores = torch.matmul(decoder_output_reshaped, encoder_output_reshaped.transpose(0, 1))  # 점수 계산
            attention_weights = F.softmax(attention_scores, dim=-1)  # 가중치 계산
            
            context = torch.matmul(attention_weights, encoder_output_reshaped)  # 문맥 벡터
            context = context.unsqueeze(1)  # 차원 확장
            
            combined = torch.cat([decoder_output, context], dim=-1)  # 결합
            projection = self.project(combined)  # 프로젝션
            
            outputs.append(projection)  # 결과 추가
            decoder_input = torch.LongTensor([targets[i]]).to(inputs.device)  # 다음 입력 (Teacher Forcing)
        
        outputs = torch.stack(outputs).squeeze()  # 텐서 결합
        return outputs  # 최종 출력 반환
    
    def translate(self, input_text, max_length=20):  # 번역 메서드
        input_seq = torch.LongTensor(list(map(ord, input_text)))  # 입력 문자열을 텐서로
        print(f"입력 시퀀스: {input_seq}")  # 확인용 출력
        
        # 인코더
        initial_state = self._init_state(1)  # 인코더 초기 상태
        embedding = self.embedding(input_seq).unsqueeze(1)  # 인코더 임베딩
        encoder_output, encoder_state = self.encoder(embedding, initial_state)  # 인코더 실행
        
        # 디코더
        decoder_state = encoder_state  # 디코더 초기 상태
        decoder_input = torch.LongTensor([ord('V')])  # 시작 토큰
        
        translated_text = []  # 번역 텍스트 저장용
        last_char = None  # 중복 문자 확인용
        repeat_count = 0  # 반복 카운트
        
        for i in range(max_length):  # 최대 길이 반복
            decoder_input = self.embedding(decoder_input).unsqueeze(1)  # 디코더 임베딩
            decoder_output, decoder_state = self.decoder(decoder_input, decoder_state)  # 디코더 실행
            
            # 어텐션 계산
            decoder_output_reshaped = decoder_output.squeeze(1)  # 디코더 출력 정렬
            encoder_output_reshaped = encoder_output.squeeze(1)  # 인코더 출력 정렬
            
            attention_scores = torch.matmul(decoder_output_reshaped, encoder_output_reshaped.transpose(0, 1))  # 점수 계산
            attention_weights = F.softmax(attention_scores, dim=-1)  # 가중치 계산
            
            context = torch.matmul(attention_weights, encoder_output_reshaped)  # 문맥 벡터
            context = context.unsqueeze(1)  # 차원 확장
            
            combined = torch.cat([decoder_output, context], dim=-1)  # 결합
            projection = self.project(combined)  # 프로젝션
            
            # 소프트맥스 적용 및 예측
            probs = F.softmax(projection.squeeze(0), dim=-1)  # 확률 계산
            values, indices = torch.topk(probs, k=1)  # 가장 높은 확률 문자 선택
            next_char = indices.item()  # 문자 코드 추출
            
            print(f"Step {i}:")  # 현재 단계
            print(f"  Projection shape: {projection.size()}")  # 차원
            print(f"  Values: {values.detach().tolist()}")  # 확률값
            print(f"  Indices: {indices.detach().tolist()}")  # 인덱스
            print(f"  Predicted char: {chr(next_char)} (ASCII: {next_char})")  # 예측 결과
            
            # 종료 조건 체크
            if next_char < 32 or next_char > 126:  # 유효 범위 밖
                break
                
            if next_char == last_char:  # 중복 문자 체크
                repeat_count += 1
                if repeat_count > 2:  # 반복 심화 시 종료
                    break
            else:
                repeat_count = 0
                last_char = next_char
            
            # 문장 종료 조건 체크
            if len(translated_text) > 0 and next_char == ord('.'):  # 마침표 등장 시
                break
                
            translated_text.append(chr(next_char))  # 결과 리스트 추가
            decoder_input = torch.LongTensor([next_char])  # 다음 단계 입력
        
        result = ''.join(translated_text)  # 문자열 결합
        print(f"최종 번역 결과: {result}")  # 최종 출력
        return result
    
    def _init_state(self, batch_size=1):  # 초기 은닉 상태
        weight = next(self.parameters()).data
        return weight.new(self.n_layers, batch_size, self.hidden_size).zero_()

print("모델 클래스 정의 완료")

## 3. 모델 초기화 및 학습 설정

In [ ]:
# 모델 초기화
seq2seq = Seq2Seq(vocab_size, 64)  # Seq2Seq 모델 인스턴스 (은닉층 64)
criterion = nn.CrossEntropyLoss()  # 손실 함수 정의
optimizer = torch.optim.Adam(seq2seq.parameters(), lr=1e-3)  # 최적화 도구
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)  # 학습률 스케줄러

print(seq2seq)  # 모델 구조 출력

## 4. 학습 (2000 에포크)

In [ ]:
log = []  # 로그 저장용
for epoch in range(2000):  # 2000 에포크 반복
    total_loss = 0  # 에포크 총 손실
    for x_text, y_text in training_pairs:  # 데이터 쌍 반복
        x = torch.LongTensor(list(map(ord, x_text)))  # 입력 텐서화
        y = torch.LongTensor(list(map(ord, y_text)))  # 타겟 텐서화
        
        prediction = seq2seq(x, y)  # 예측
        loss = criterion(prediction, y)  # 손실 계산
        optimizer.zero_grad()  # 기울기 초기화
        loss.backward()  # 역전파
        torch.nn.utils.clip_grad_norm_(seq2seq.parameters(), 1.0)  # 기울기 클리핑
        optimizer.step()  # 업데이트
        total_loss += loss.item()  # 손실 누적
    
    avg_loss = total_loss / len(training_pairs)  # 평균 손실
    scheduler.step(avg_loss)  # 스케줄러 업데이트
    log.append(avg_loss)  # 로그 기록
    
    if epoch % 100 == 0:  # 100회마다 출력
        print(f"\nEpoch: {epoch}, Average Loss: {avg_loss:.4f}")  # 현황
        for x_text, y_text in training_pairs:  # 각 데이터 결과 확인
            x = torch.LongTensor(list(map(ord, x_text)))
            y = torch.LongTensor(list(map(ord, y_text)))
            prediction = seq2seq(x, y)
            _, top1 = prediction.data.topk(1, 1)
            print(f"  Input: {x_text}")
            print(f"  Prediction: {[chr(c) for c in top1.squeeze().tolist()]}")

## 5. 손실 그래프

In [ ]:
plt.figure(figsize=(10, 4))  # 그래프 크기
plt.plot(log)  # 로그 플롯
plt.xlabel('Epoch')  # x축
plt.ylabel('Cross Entropy Loss')  # y축
plt.title('Training Loss')  # 제목
plt.grid(True)  # 격자
plt.show()  # 출력

## 6. 번역 테스트

In [ ]:
print("번역 테스트:")  # 출력
test_texts = ["I go to bed", "I want to sleep", "I play a game"]  # 테스트 리스트
for text in test_texts:  # 반복
    print(f"\n입력: {text}")  # 입력 문장
    translated_text = seq2seq.translate(text)  # 번역 결과
    print(f"번역: {translated_text}")  # 결과 출력